In [1]:
from tokenizers import Tokenizer      #holds the tokenizers
from tokenizers.models import BPE           # the actual BPE models
from tokenizers.trainers import BpeTrainer         # learns the vocabulary + merge rules
from tokenizers.pre_tokenizers import ByteLevel       # handles word boundries before bpe
from pathlib import Path                                #handle file paths
from tokenizers.decoders import ByteLevel as ByteLevelDecoder       #decodes byte-level tokens

In [2]:
#dataset paths
train_dir = Path("/srv/data/lt2326-h26/a1/train")
languages = ["en", "tr", "zh"]

In [3]:
#training files
train_files = []

for lang in languages:
   path = train_dir / f"{lang}.txt"
   train_files.append(str(path))             #add each language file to the list of training files

In [4]:
#creating BPE-10k tokenizer

tokenizer = Tokenizer(BPE(unk_token="<UNK>"))    #tokenizer object with BPE model and unknown token but it is not trained yet 
tokenizer.pre_tokenizer = ByteLevel()               #use byte-level tokenization before BPE

In [5]:
#creating BPE trainer
trainer = BpeTrainer(
    vocab_size=10000,      #building a vocab of about 10k tokens
    special_tokens=["<UNK>"],           #add unknown token to the vocab
    initial_alphabet=ByteLevel.alphabet()           #include complete byte-level alphabet in the vocab
)


In [6]:
tokenizer.train(train_files, trainer)     #train tokenizer on combined EN + TR + ZH data


In [7]:
#checking vocabulary size and first 15 tokens

vocab = tokenizer.get_vocab()

print("Vocabulary size:", tokenizer.get_vocab_size())
print("First 30 vocabulary tokens:")
vocab_by_id = sorted(vocab.items(), key=lambda x: x[1])     # sort vocabulary by token ID
for token, token_id in vocab_by_id[:15]: # print first 15 tokens
    print(f"ID: {token_id},Token: {token}")

Vocabulary size: 10000
First 30 vocabulary tokens:
ID: 0,Token: <UNK>
ID: 1,Token: !
ID: 2,Token: "
ID: 3,Token: #
ID: 4,Token: $
ID: 5,Token: %
ID: 6,Token: &
ID: 7,Token: '
ID: 8,Token: (
ID: 9,Token: )
ID: 10,Token: *
ID: 11,Token: +
ID: 12,Token: ,
ID: 13,Token: -
ID: 14,Token: .


In [8]:
#Encode /Decode test

tokenizer.decoder = ByteLevelDecoder()   #add a decoder to the tokenizer so that we can decode the tokenized text back to original text

#encode / decode test
test_texts = [
    "She played in two matches at this championship.",
    "Germantown akademisine devam ettikten sonra 1869-1870 yılları Fransa ve Almanya'da okullarda geçti.",
    "，也是一所专门提供研究生及以上级别课程的私立精英级大学。"
]

for text in test_texts:
    encoding = tokenizer.encode(text)           # convert text into token IDS
    decoded = tokenizer.decode(encoding.ids)        #convert token IDS back to text

    print("\n========== Encode / Decode Test ==========")
    print("Original:", text)
    print("Tokens:", encoding.tokens)
    print("Decoded:", decoded)
    print("Match:", text == decoded.lstrip())  # Remove trailing whitespace for comparison


========== Encode / Decode Test ==========
Original: She played in two matches at this championship.
Tokens: ['ĠShe', 'Ġplayed', 'Ġin', 'Ġtwo', 'Ġmat', 'ches', 'Ġat', 'Ġthis', 'Ġch', 'ampionship', '.']
Decoded:  She played in two matches at this championship.
Match: True

========== Encode / Decode Test ==========
Original: Germantown akademisine devam ettikten sonra 1869-1870 yılları Fransa ve Almanya'da okullarda geçti.
Tokens: ['ĠGerman', 't', 'own', 'Ġak', 'adem', 'isine', 'Ġdevam', 'Ġett', 'ikten', 'Ġsonra', 'Ġ18', '69', '-', '18', '70', 'ĠyÄ±llarÄ±', 'ĠFransa', 'Ġve', 'ĠAlmanya', "'d", 'a', 'Ġokul', 'larda', 'ĠgeÃ§ti', '.']
Decoded:  Germantown akademisine devam ettikten sonra 1869-1870 yılları Fransa ve Almanya'da okullarda geçti.
Match: True

========== Encode / Decode Test ==========
Original: ，也是一所专门提供研究生及以上级别课程的私立精英级大学。
Tokens: ['Ġï¼Į', 'ä¹Ł', 'æĺ¯ä¸Ģ', 'æīĢ', 'ä¸ĵ', 'éĹ¨', 'æıĲä¾Ľ', 'çłĶç©¶', 'çĶŁ', 'åıĬ', 'ä»¥ä¸Ĭ', 'çº§', 'åĪ«', 'è¯¾', 'ç¨ĭ', 'çļĦ', 'ç§ģ', 'ç«ĭ', 'ç²¾', '

In [9]:
#validation statistics

valid_dir = Path("/srv/data/lt2326-h26/a1/valid")


#example tokenization on validation data
for lang in languages:
    path = valid_dir / f"{lang}.txt"

    text = path.read_text(encoding="utf-8")
    lines= text.splitlines()

    sentence = lines[0] #take the first sentence from the validation file for each language 
    encoding = tokenizer.encode(sentence)
    tokens = encoding.tokens

    print(f"\n========== {lang.upper()} ==========")
    print("Original text:", sentence)
    print("BPE Tokens:", tokens)
    print("Number of BPE Tokens:", len(tokens))



========== EN ==========
Original text: She played in two matches at this championship.
BPE Tokens: ['ĠShe', 'Ġplayed', 'Ġin', 'Ġtwo', 'Ġmat', 'ches', 'Ġat', 'Ġthis', 'Ġch', 'ampionship', '.']
Number of BPE Tokens: 11

========== TR ==========
Original text: Germantown akademisine devam ettikten sonra 1869-1870 yılları Fransa ve Almanya'da okullarda geçti.
BPE Tokens: ['ĠGerman', 't', 'own', 'Ġak', 'adem', 'isine', 'Ġdevam', 'Ġett', 'ikten', 'Ġsonra', 'Ġ18', '69', '-', '18', '70', 'ĠyÄ±llarÄ±', 'ĠFransa', 'Ġve', 'ĠAlmanya', "'d", 'a', 'Ġokul', 'larda', 'ĠgeÃ§ti', '.']
Number of BPE Tokens: 25

========== ZH ==========
Original text: ，也是一所专门提供研究生及以上级别课程的私立精英级大学。
BPE Tokens: ['Ġï¼Į', 'ä¹Ł', 'æĺ¯ä¸Ģ', 'æīĢ', 'ä¸ĵ', 'éĹ¨', 'æıĲä¾Ľ', 'çłĶç©¶', 'çĶŁ', 'åıĬ', 'ä»¥ä¸Ĭ', 'çº§', 'åĪ«', 'è¯¾', 'ç¨ĭ', 'çļĦ', 'ç§ģ', 'ç«ĭ', 'ç²¾', 'èĭ±', 'çº§', 'å¤§åŃ¦', 'ãĢĤ']
Number of BPE Tokens: 23


In [10]:
#validation statistics for each language
for lang in languages:
    path = valid_dir / f"{lang}.txt"
    text = path.read_text(encoding="utf-8")
    lines= text.splitlines()

    total_bpe_tokens = 0
    for sentence in lines:
        encoding = tokenizer.encode(sentence)
        bpe_tokens = encoding.tokens
        total_bpe_tokens += len(bpe_tokens)         #count BPE tokens for each sentence and add to total

    total_characters = sum(len(sentence) for sentence in lines)         #count Unicode characters in each sentence and add to total

    print(f"\n========== {lang.upper()} ==========")
    print("Total number of BPE tokens:", total_bpe_tokens)
    print("Number of sentences:", len(lines))
    print("Total unicode characters:", total_characters)
    average_bpe_tokens = total_bpe_tokens / len(lines)
    print("Average number of BPE tokens per sentence:", average_bpe_tokens)
    print("BPE tokens per Unicode character:", total_bpe_tokens / total_characters)



========== EN ==========
Total number of BPE tokens: 114312
Number of sentences: 4540
Total unicode characters: 368269
Average number of BPE tokens per sentence: 25.17885462555066
BPE tokens per Unicode character: 0.31040353654529707

========== TR ==========
Total number of BPE tokens: 113376
Number of sentences: 3735
Total unicode characters: 368383
Average number of BPE tokens per sentence: 30.355020080321285
BPE tokens per Unicode character: 0.30776664504062345

========== ZH ==========
Total number of BPE tokens: 299545
Number of sentences: 9554
Total unicode characters: 368266
Average number of BPE tokens per sentence: 31.352836508268787
BPE tokens per Unicode character: 0.8133930365550988


In [11]:
#Encode / Decode and UNK sanity check

print("=" * 10 + "Encode/ Decode and UNK sanity check" + "=" * 10)

test_texts = "Hello World!"

encoding = tokenizer.encode(test_texts)
print("Original text:", test_texts)
print("BPE Tokens:", encoding.tokens)
print("Token IDs:", encoding.ids)

decoded = tokenizer.decode(encoding.ids)
print("Decoded text:", decoded)
print("Match:", test_texts == decoded.lstrip())  # Remove trailing whitespace for comparison

#checking whether the tokenizer produced <UNK>
unk_id = vocab.get("<UNK>")

contains_unk = unk_id in encoding.ids
print("Contains <UNK> token:", contains_unk)


==========Encode/ Decode and UNK sanity check==========
Original text: Hello World!
BPE Tokens: ['ĠH', 'ell', 'o', 'ĠWorld', '!']
Token IDs: [422, 1185, 79, 3682, 1]
Decoded text:  Hello World!
Match: True
Contains <UNK> token: False


In [12]:
#Unseen character test

print("\n========== Unseen Character Test ==========")

test_text = "Hello 🙂"

encoding = tokenizer.encode(test_text)
print("Original text:", test_text)
print("BPE Tokens:", encoding.tokens)
print("Token IDs:", encoding.ids)

decoded = tokenizer.decode(encoding.ids)
print("Decoded text:", decoded)
print("Match:", test_text == decoded.lstrip())  # Remove trailing whitespace for comparison

#checking whether the tokenizer produced <UNK>
unk_id = vocab.get("<UNK>")

contains_unk = unk_id in encoding.ids
print("Contains <UNK> token:", contains_unk)



========== Unseen Character Test ==========
Original text: Hello 🙂
BPE Tokens: ['ĠH', 'ell', 'o', 'Ġ', 'ð', 'Ł', 'Ļ', 'Ĥ']
Token IDs: [422, 1185, 79, 221, 173, 254, 248, 225]
Decoded text:  Hello 🙂
Match: True
Contains <UNK> token: False


In [13]:
#save trained tokenizer
output_dir = Path("tokenizers")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "bpe_10k.json"

tokenizer.save(str(output_path))            #saving tokenizers as JSON file so that it can be loaded later for inference or further training

print("=" * 10 + "Saving Tokenizer" + "=" * 10)
print("Tokenizer saved to:", output_path)


==========Saving Tokenizer==========
Tokenizer saved to: tokenizers/bpe_10k.json


In [14]:
#reloading the saved tokenizer
loaded_tokenizer = Tokenizer.from_file(str(output_path))

print("=" * 10 + "Reloading Test" + "=" * 10)

test_text = "She played in two matches at this championship."

encoding = loaded_tokenizer.encode(test_text)
decoded = loaded_tokenizer.decode(encoding.ids)

print("Original text:", test_text)
print("BPE Tokens:", encoding.tokens)
print("Decoded text:", decoded)
print("Match:", test_text == decoded.lstrip())  # checking whether the decoded text matches the original text after reloading the tokenizer


==========Reloading Test==========
Original text: She played in two matches at this championship.
BPE Tokens: ['ĠShe', 'Ġplayed', 'Ġin', 'Ġtwo', 'Ġmat', 'ches', 'Ġat', 'Ġthis', 'Ġch', 'ampionship', '.']
Decoded text:  She played in two matches at this championship.
Match: True
